# Google Search Ranking & Discoverability Capstone (ML)

**Status: BUILT. Not DATA-VALIDATED. No results generated.**

Provisional lane: **Refresh / Content Opportunity Scoring**. This is a candidate only; the lane is final only after the real schema is inspected (Step 2).

Public rule: no client names, domains, URLs, private queries, credentials, raw exports, or claims that prove Google's algorithm or causal refresh impact.

## Step 0. Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "flyrank_ml").exists())
sys.path.insert(0, str(ROOT / "src"))

from flyrank_ml.config import BlockedError, get_hf_token, load_config, load_env_file, require_configured
load_env_file()          # reads HF_TOKEN from your local .env (never paste the token here)
cfg = load_config()
print("Project root:", ROOT)
print("HF_TOKEN set:", bool(get_hf_token()))

## Step 1. Inspect the real warehouse schema
Set `hf_glob` in `configs/data_config.yaml` from the FlyRank warehouse resource card first.

In [ ]:
from flyrank_ml.data import connect_duckdb, inspect_schema

try:
    if not cfg["data"].get("hf_glob"):
        raise BlockedError("Set hf_glob in configs/data_config.yaml.")
    con = connect_duckdb()
    schema = inspect_schema(con, cfg["data"]["hf_glob"])
    for path, cols in schema.items():
        print(path, [c["column"] for c in cols])
except BlockedError as exc:
    print("BLOCKED:", exc)

## Step 2. Lane decision gate
Refresh / Content Opportunity Scoring is supported only if the warehouse has, at page level: a page identifier, a daily date, clicks, impressions, and enough history for two 28-day windows plus two 28-day label horizons (at least about 120 days).
If any is missing, pick the best-supported lane and record why in `docs/METHODOLOGY.md`. Then map the columns in `configs/data_config.yaml`.

## Step 3. Load aggregated data and check quality

In [ ]:
from flyrank_ml.data.loader import load_daily_from_warehouse

try:
    daily = load_daily_from_warehouse(cfg["data"])
    print("Loaded aggregated rows:", len(daily))
except BlockedError as exc:
    daily = None
    print("BLOCKED:", exc)

In [ ]:
from flyrank_ml.data import quality_report, clean_daily
import json

if daily is not None:
    q = quality_report(daily, "page_id", "date", ["clicks", "impressions"])
    print(json.dumps(q, indent=2))
    clean, exclusions = clean_daily(daily, "page_id", "date", ["clicks", "impressions"])
    print("Exclusions:", exclusions)

## Step 4. Features, label, baseline, model, validation
Features: two equal windows ending at the decision date. Label: decline of at least 30% in the next 28 days vs the last 28 days (pages with at least 10 recent clicks). Split: held-out pages and a later period. Leakage guards raise errors.

In [ ]:
from flyrank_ml.pipeline import run_pipeline

if daily is not None:
    exp = {**cfg["experiment"], "algorithm": cfg["model"]["algorithm"], "model_params": cfg["model"]["model_params"]}
    exp.pop("lane", None); exp.pop("lane_status", None)
    results = run_pipeline(daily, ROOT / "reports", data_source="flyrank_real", cfg=exp)
    print("Cutoffs:", results["cutoffs"], "| train:", results["n_train"], "valid:", results["n_valid"])
    print("Model   :", {k: round(v, 3) for k, v in results["model_metrics"].items() if isinstance(v, float)})
    print("Baseline:", {k: round(v, 3) for k, v in results["baseline_metrics"].items() if isinstance(v, float)})
    print("AUC diff bootstrap:", results["roc_auc_diff_bootstrap_model_minus_baseline"])
else:
    print("Skipped: BLOCKED until real data is available.")

## Step 5. Ranked recommendations (pseudonymous IDs, reason codes)

In [ ]:
import pandas as pd
p = ROOT / "reports" / "tables" / "ranked_recommendations.csv"
if p.exists():
    print(pd.read_csv(p).head(10)[["rank", "page_pseudonym", "action", "reason_codes", "score"]])
else:
    print("No recommendations yet.")

## Step 6. Build the paper
Run `python scripts/build_paper.py`, then follow `docs/GITHUB_UPLOAD_GUIDE.md`.

## Limitations (always state)
Scores estimate decline risk from historical metrics. They are decision-support signals, not confirmed outcomes, and say nothing about causal effects of refreshing or how Google ranks pages.